# AIML Computer Vision Case Study: CIFAR-100

This beginner-friendly notebook trains and compares a custom convolutional neural network (CNN) and Keras VGG-16. It reports test accuracy, plots training and validation curves, and saves model summaries and architecture diagrams.

**Run in Google Colab:** choose **Runtime → Change runtime type → T4 GPU**, then use **Runtime → Run all**. Training time depends on the GPU; the callback stops early if validation loss stops improving. The VGG-16 model uses ImageNet pretrained features for a practical CIFAR-100 transfer-learning baseline.

> Run the cells from top to bottom. The notebook downloads CIFAR-100 automatically. Keep the generated `.png` diagrams and `.keras` checkpoints alongside this notebook when preparing your submission.

## 1. Imports and reproducibility

TensorFlow/Keras provides the dataset, models, callbacks, and plotting helpers. A fixed seed makes the split and initialization more repeatable.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__)
print('Available GPUs:', tf.config.list_physical_devices('GPU'))

## 2. Load and preprocess CIFAR-100

CIFAR-100 contains 50,000 training and 10,000 test color images, each 32×32, across 100 fine-grained classes. Pixel values are converted to floats in [0, 1]. We hold out 5,000 training images for validation; the test set is kept for final evaluation only. VGG-16 expects larger inputs, so a dataset pipeline resizes images to 96×96 and applies VGG's ImageNet preprocessing.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar100.load_data(label_mode='fine')
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
y_train = y_train.squeeze()
y_test = y_test.squeeze()

# Deterministic validation split: 45,000 train / 5,000 validation.
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(x_train))
val_idx, train_idx = indices[:5000], indices[5000:]
x_val, y_val = x_train[val_idx], y_train[val_idx]
x_train, y_train = x_train[train_idx], y_train[train_idx]

NUM_CLASSES = 100
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(images, labels, training=False, vgg=False):
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if training:
        ds = ds.shuffle(10000, seed=SEED, reshuffle_each_iteration=True)
    def prepare(image, label):
        if vgg:
            image = tf.image.resize(image, (96, 96))
            image = image * 255.0
            image = keras.applications.vgg16.preprocess_input(image)
        return image, tf.cast(label, tf.int32)
    return ds.map(prepare, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_ds(x_train, y_train, training=True)
val_ds = make_ds(x_val, y_val)
test_ds = make_ds(x_test, y_test)
vgg_train_ds = make_ds(x_train, y_train, training=True, vgg=True)
vgg_val_ds = make_ds(x_val, y_val, vgg=True)
vgg_test_ds = make_ds(x_test, y_test, vgg=True)
print('Train:', x_train.shape, '| Validation:', x_val.shape, '| Test:', x_test.shape)

## 3. Build the custom CNN

This architecture has three convolution blocks. It uses He initialization with ReLU, batch normalization to stabilize learning, max pooling to reduce image size, dropout to reduce overfitting, and L2 kernel regularization. These choices demonstrate the requested architecture experiments in one clear baseline.

In [ ]:
def build_custom_cnn():
    inputs = keras.Input(shape=(32, 32, 3), name='image')
    x = inputs
    for filters, drop_rate in [(32, 0.20), (64, 0.30), (128, 0.40)]:
        x = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal',
                         kernel_regularizer=keras.regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal',
                         kernel_regularizer=keras.regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(drop_rate)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu', kernel_initializer='he_normal',
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.50)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='class_probabilities')(x)
    return keras.Model(inputs, outputs, name='custom_cnn')

custom_model = build_custom_cnn()
custom_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                     loss='sparse_categorical_crossentropy', metrics=['accuracy'])
custom_model.summary()

In [ ]:
# Save the architecture diagram and the full model summary as submission artifacts.
keras.utils.plot_model(custom_model, to_file='custom_cnn_architecture.png',
                       show_shapes=True, show_layer_names=True, dpi=100)
with open('custom_cnn_summary.txt', 'w') as f:
    custom_model.summary(print_fn=lambda line: f.write(line + '\n'))

## 4. Train the custom CNN

EarlyStopping restores the best validation-loss weights. ModelCheckpoint keeps the best full model on disk, so it can be loaded later even if the runtime disconnects.

In [ ]:
EPOCHS_CUSTOM = 30
custom_callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint('best_custom_cnn.keras', monitor='val_loss',
                                    save_best_only=True, verbose=1)
]
history_custom = custom_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_CUSTOM,
                                  callbacks=custom_callbacks)

## 5. Build and train VGG-16

VGG-16's convolutional backbone is initialized with ImageNet weights and frozen for the first training pass. A small classifier head learns the 100 CIFAR-100 categories. Resizing to 96×96 retains more visual detail than resizing to the minimum input size while remaining manageable on a Colab GPU.

> If the pretrained-weight download is unavailable, replace `weights='imagenet'` with `weights=None`. That trains VGG-16 from scratch and will usually need more time and epochs.

In [ ]:
def build_vgg16():
    base = keras.applications.VGG16(include_top=False, weights='imagenet',
                                    input_shape=(96, 96, 3))
    base.trainable = False  # transfer learning: train the new head first
    inputs = keras.Input(shape=(96, 96, 3), name='image')
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.40)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='class_probabilities')(x)
    return keras.Model(inputs, outputs, name='vgg16_cifar100')

vgg_model = build_vgg16()
vgg_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
vgg_model.summary()

In [ ]:
keras.utils.plot_model(vgg_model, to_file='vgg16_architecture.png',
                       show_shapes=True, show_layer_names=True, dpi=100)
with open('vgg16_summary.txt', 'w') as f:
    vgg_model.summary(print_fn=lambda line: f.write(line + '\n'))

In [ ]:
EPOCHS_VGG = 12
vgg_callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint('best_vgg16.keras', monitor='val_loss',
                                    save_best_only=True, verbose=1)
]
history_vgg = vgg_model.fit(vgg_train_ds, validation_data=vgg_val_ds, epochs=EPOCHS_VGG,
                            callbacks=vgg_callbacks)

## 6. Evaluate both models on the test set

Accuracy is measured on the held-out test set after model selection on the validation set. The exact results will appear after you run training.

In [ ]:
custom_test_loss, custom_test_acc = custom_model.evaluate(test_ds, verbose=1)
vgg_test_loss, vgg_test_acc = vgg_model.evaluate(vgg_test_ds, verbose=1)

comparison = pd.DataFrame({
    'Model': ['Custom CNN', 'VGG-16 (ImageNet transfer learning)'],
    'Test loss': [custom_test_loss, vgg_test_loss],
    'Test accuracy (%)': [100 * custom_test_acc, 100 * vgg_test_acc]
})
display(comparison.style.format({'Test loss': '{:.4f}', 'Test accuracy (%)': '{:.2f}%'}))
comparison.to_csv('test_accuracy_comparison.csv', index=False)

## 7. Plot training and validation curves

These curves compare train and validation accuracy and loss for both models. Validation curves are plotted as the held-out data curves because a separate validation split was used during training; test data remains reserved for the final accuracy comparison.

In [ ]:
def plot_histories(histories):
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    metrics = [('accuracy', 'Accuracy'), ('loss', 'Loss')]
    for row, (metric, label) in enumerate(metrics):
        for history, name in histories:
            axes[row, 0].plot(history.history[metric], label=f'{name} train')
            axes[row, 0].plot(history.history['val_' + metric], linestyle='--', label=f'{name} validation')
        axes[row, 0].set_title(f'{label}: model comparison')
        axes[row, 0].set_xlabel('Epoch'); axes[row, 0].set_ylabel(label); axes[row, 0].legend(); axes[row, 0].grid(True, alpha=.3)
        for history, name in histories:
            axes[row, 1].plot(history.history[metric], label=f'{name} train')
            axes[row, 1].plot(history.history['val_' + metric], linestyle='--', label=f'{name} validation')
        axes[row, 1].set_title(f'{label}: learning curves')
        axes[row, 1].set_xlabel('Epoch'); axes[row, 1].set_ylabel(label); axes[row, 1].legend(); axes[row, 1].grid(True, alpha=.3)
    plt.tight_layout()
    plt.savefig('training_validation_curves.png', dpi=160, bbox_inches='tight')
    plt.show()

plot_histories([(history_custom, 'Custom CNN'), (history_vgg, 'VGG-16')])

## 8. View architecture diagrams

The diagrams are also saved to the current Colab working directory for download and GitHub submission.

In [ ]:
from IPython.display import Image, display
display(Image('custom_cnn_architecture.png'))
display(Image('vgg16_architecture.png'))

## 9. GitHub submission checklist

- [ ] Run all cells in Google Colab with a GPU and confirm there are no errors.
- [ ] Confirm the final test-accuracy table contains results for both models.
- [ ] Save the executed notebook with outputs (`.ipynb`).
- [ ] Include `custom_cnn_architecture.png`, `vgg16_architecture.png`, `training_validation_curves.png`, and both `.txt` model summaries.
- [ ] Include `test_accuracy_comparison.csv`; include the two `.keras` checkpoints if repository size limits allow.
- [ ] Add a short `README.md` with the project objective, Colab run instructions, and your observed results.
- [ ] Create a GitHub repository, upload/commit the notebook and supporting files, and verify the notebook opens on GitHub.
- [ ] Submit the repository URL through Paatshala as required by the case study.

**Record your results here after running:** Custom CNN test accuracy: ______%; VGG-16 test accuracy: ______%.
